In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 
import math 

from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score


# Đọc dữ liệu 

Dữ liệu về giá nhà ở Boston được hỗ trợ bởi sklearn, đọc dữ liệu thông qua hàm `datasets.load_boston()` 

Xem thêm các bộ dữ liệu khác tại https://scikit-learn.org/stable/datasets/index.html#toy-datasets. 

Dữ liệu được chia thành các thành phần data và target như tập diabetes. Dữ liệu cũng đã được chuẩn hóa, chỉ cần gọi ra và huấn luyện

In [ ]:
# lay du lieu dataset - du lieu ve giá nhà
try:
    dataset = datasets.load_boston()
except ImportError:
    # scikit-learn >= 1.2 đã remove load_boston(); đọc lại từ file CSV local đi kèm sklearn (không cần internet)
    import sklearn
    from pathlib import Path
    from sklearn.utils import Bunch
    csv_path = (
        Path(sklearn.__file__).resolve().parent
        / "datasets" / "data" / "boston_house_prices.csv"
    )
    raw_df = pd.read_csv(csv_path, skiprows=1)
    raw_df.columns = [c.replace("\n", "") for c in raw_df.columns]
    data = raw_df.drop(columns=["MEDV"]).to_numpy()
    target = raw_df["MEDV"].to_numpy()
    dataset = Bunch(data=data, target=target)

print("Số chiều dữ liệu input: ", dataset.data.shape)
print("Số chiều dữ liệu target: ", dataset.target.shape)
print()

print("5 mẫu dữ liệu đầu tiên:")
print("input: ", dataset.data[:5])
print("target: ",dataset.target[:5])

URLError: <urlopen error [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond>

**Chia dữ liệu làm 2 phần training 362 mẫu và testing 80 mẫu** 

In [19]:
# cat nho du lieu, lay 1 phan cho qua trinh thu nghiem,
# chia train test cac mau du lieu
# dataset_X = dataset.data[:, np.newaxis, 2]
dataset_X = dataset.data

dataset_X_train = dataset_X[:404]
dataset_y_train = dataset.target[:404]

dataset_X_test = dataset_X[405:]
dataset_y_test = dataset.target[405:]

# Xây dựng mô hình

## Xây dựng mô hình bằng thư viện

In [20]:
regr = linear_model.LinearRegression()

## Xây dựng mô hình Linear Regression tự viết 

In [21]:
def linear_regression(dataset_X_train,dataset_y_train):
    one = np.ones((dataset_X_train.shape[0],1))
    Xbar = np.concatenate((one, dataset_X_train), axis = 1)

    A = np.dot(Xbar.T, Xbar)
    b = np.dot(Xbar.T, dataset_y_train)
    w_lr = np.dot(np.linalg.pinv(A), b)
    coef = w_lr[1:]
    intercept = w_lr[0]
    return coef, intercept

## Hàm test mô hình tự viết

In [22]:
def predict(intercept, coef, dataset_X_test):
    y_pred_ = [intercept] + coef.dot(dataset_X_test[0].T)
    for i in range(1, len(dataset_X_test)):
        y_pred = intercept + coef.dot(dataset_X_test[i].T)
        y_pred_= np.append(y_pred_,y_pred)
    return y_pred_

# Huấn luyện mô hình

## Huấn luyện mô hình của thư viện

In [33]:
regr.fit(dataset_X_train, dataset_y_train)
print("[w1, ... w_n] = ", regr.coef_)
print("w0 = ", regr.intercept_)

[w1, ... w_n] =  [-2.02135297e-01  4.41276341e-02  5.26739364e-02  1.88474315e+00
 -1.49281487e+01  4.76038673e+00  2.88734527e-03 -1.30025278e+00
  4.61661953e-01 -1.55434673e-02 -8.11632369e-01 -1.97174433e-03
 -5.32273431e-01]
w0 =  30.077166922901856


## Training mô hình bằng Linear regression tự viết

In [24]:
coef, intercept = linear_regression(dataset_X_train,dataset_y_train)
print("[w1, ... w_n] = ", coef)
print("w0 = ", intercept)

[w1, ... w_n] =  [-2.02135297e-01  4.41276341e-02  5.26739364e-02  1.88474315e+00
 -1.49281487e+01  4.76038673e+00  2.88734527e-03 -1.30025278e+00
  4.61661953e-01 -1.55434673e-02 -8.11632369e-01 -1.97174433e-03
 -5.32273431e-01]
w0 =  30.07716691924543


# Dự đoán các mẫu dữ liệu

## Dự đoán các mẫu dữ liệu theo mô hình của thư viện

In [25]:
dataset_y_pred_lib = regr.predict(dataset_X_test)
pd.DataFrame(data=np.array([dataset_y_test, dataset_y_pred_lib,
                            abs(dataset_y_test - dataset_y_pred_lib)]).T,
             columns=["Thực tế", "Dự đoán", "Lệch"])  

,Thực tế,Dự đoán,Lệch
0,5.0,3.787057,1.212943
1,11.9,6.640550,5.259450
2,27.9,21.312765,6.587235
3,17.2,15.412714,1.787286
4,27.5,23.652298,3.847702
...,...,...,...
96,22.4,23.755044,1.355044
97,20.6,22.081673,1.481673
98,23.9,28.181773,4.281773
99,22.0,26.572420,4.572420


## Dự đoán các mẫu dữ liệu tính theo linear regression tự viết

In [26]:
dataset_y_pred = predict(intercept, coef, dataset_X_test)
pd.DataFrame(data=np.array([dataset_y_test, dataset_y_pred,
                            abs(dataset_y_test - dataset_y_pred)]).T,
             columns=["Thực tế", "Dự đoán", "Lệch"]) 

,Thực tế,Dự đoán,Lệch
0,5.0,3.787057,1.212943
1,11.9,6.640550,5.259450
2,27.9,21.312765,6.587235
3,17.2,15.412714,1.787286
4,27.5,23.652298,3.847702
...,...,...,...
96,22.4,23.755044,1.355044
97,20.6,22.081673,1.481673
98,23.9,28.181773,4.281773
99,22.0,26.572420,4.572420


## Đánh giá mô hình linear regression của thư viện

In [27]:
loss = math.sqrt(mean_squared_error(dataset_y_test, dataset_y_pred_lib))
print("lỗi :",loss)

lỗi : 5.749521870254025


## Đánh giá mô hình linear regression tự viết

In [28]:
loss = math.sqrt(mean_squared_error(dataset_y_test, dataset_y_pred))
print("lỗi :",loss)

lỗi : 5.749521870168214
